# Fine-tuning LoRA du tuteur — style adapté au niveau de l'étudiant

**Objectif** (cahier des charges : *fine-tuning a Hugging Face model to produce your desired tutoring style*) :
apprendre à **Qwen/Qwen2.5-1.5B-Instruct** à répondre comme le tuteur de l'application, en adaptant son
explication au niveau indiqué (débutant, intermédiaire, avancé), à partir d'un passage du cours, et en citant
la page sous la forme `[p.X]`.

**Méthode : QLoRA**. Le modèle de base est chargé en 4 bits (gelé) ; on n'entraîne que de petites matrices
LoRA ajoutées aux couches d'attention et MLP (environ 1 % des paramètres). Cela tient dans les 15 Go du GPU T4
gratuit de Colab.

**Étapes**
1. Installer les bibliothèques et vérifier le GPU.
2. Envoyer `train.jsonl` et `test.jsonl` (produits en local par `python -m app.phase4_finetuning.dataset`).
3. Charger le modèle en 4 bits, configurer LoRA, entraîner avec `SFTTrainer`.
4. Tracer la courbe de perte, sauvegarder l'adaptateur.
5. Générer les réponses du modèle de **base** et du modèle **affiné** sur tout le jeu de test (mêmes prompts,
   même température) → `results_base.jsonl` et `results_finetuned.jsonl`, à évaluer en local
   avec `python -m app.phase4_finetuning.evaluate`.
6. Publier l'adaptateur sur le Hugging Face Hub avec une model card.

> Menu **Exécution → Modifier le type d'exécution → GPU T4** avant de commencer.

## 1. Installation

`transformers` (modèle), `peft` (LoRA), `trl` (`SFTTrainer`), `bitsandbytes` (quantification 4 bits),
`datasets` (lecture des JSONL) et `accelerate` (placement sur le GPU). Compter 1 à 2 minutes.

In [ ]:
!pip install -q -U "transformers>=4.56" "peft>=0.15" "trl>=0.19" "bitsandbytes>=0.45" datasets accelerate

In [ ]:
import torch, transformers, peft, trl
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (changer le type d'exécution)")
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "Go")
print("transformers", transformers.__version__, "· peft", peft.__version__, "· trl", trl.__version__)

## 2. Envoi du jeu de données

Sélectionner **les deux fichiers** `train.jsonl` et `test.jsonl` (dossier `backend/app/phase4_finetuning/data/`).
Chaque ligne est une conversation au format chat : `system` (règles du tuteur), `user` (niveau + passage du
cours + question), `assistant` (réponse idéale rédigée par le modèle enseignant qwen2.5:7b puis filtrée).

In [ ]:
import os
from google.colab import files

# Les deux fichiers vont toujours dans data/train.jsonl et data/test.jsonl, même si Colab les renomme
# (« train (1).jsonl » quand on les renvoie). Déjà présents : rien à renvoyer.
os.makedirs("data", exist_ok=True)
EXPECTED = ("train", "test")
if not all(os.path.exists(f"data/{name}.jsonl") for name in EXPECTED):
    for uploaded in files.upload():
        target = next((name for name in EXPECTED if uploaded.startswith(name)), None)
        if target is None:
            print(f"Fichier ignoré : {uploaded}")
            os.remove(uploaded)
            continue
        os.replace(uploaded, f"data/{target}.jsonl")
missing = [f"data/{name}.jsonl" for name in EXPECTED if not os.path.exists(f"data/{name}.jsonl")]
assert not missing, f"Fichiers manquants : {missing} — relance la cellule et envoie train.jsonl et test.jsonl."
print(sorted(os.listdir("data")))

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": "data/train.jsonl", "test": "data/test.jsonl"})
print(raw)

# Format « prompt / completion » de TRL : la perte n'est calculée que sur la réponse du tuteur,
# pas sur le prompt (règles + passage), qui est recopié et n'a rien à apprendre.
def to_prompt_completion(example):
    return {"prompt": example["messages"][:-1], "completion": example["messages"][-1:]}

train_ds = raw["train"].map(to_prompt_completion, remove_columns=raw["train"].column_names)
eval_ds = raw["test"].map(to_prompt_completion, remove_columns=raw["test"].column_names)
print(train_ds[0]["prompt"][1]["content"][:400], "…")
print("\nRéponse attendue :", train_ds[0]["completion"][0]["content"][:400])

## 3. Modèle de base en 4 bits (QLoRA)

- `nf4` + double quantification : environ 1,2 Go de VRAM pour les poids du modèle de 1,5 milliard de paramètres.
- Calculs en `float16` : le T4 ne gère pas `bfloat16` (`bf16=False` à l'étape 4).
- Les poids LoRA, eux, sont gardés en `float32` (étape 4) : en `float16`, l'entraînement en précision
  mixte échoue (« Attempting to unscale FP16 gradients »).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"  # entraînement (la génération repassera à gauche)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map={"": 0},
                                             dtype=torch.float16)
model.config.use_cache = False  # incompatible avec le gradient checkpointing pendant l'entraînement
print(f"Mémoire du modèle : {model.get_memory_footprint() / 1e9:.2f} Go")

## 4. Configuration LoRA et de l'entraînement

**LoRA** : rang `r=16`, `alpha=32` (facteur d'échelle alpha/r = 2), `dropout=0.05`, sur toutes les projections
d'attention (`q_proj`, `k_proj`, `v_proj`, `o_proj`) et du MLP (`gate_proj`, `up_proj`, `down_proj`).

**Entraînement**, réglé pour 15 Go de VRAM :
- lots de 4 exemples × 4 étapes d'accumulation = **16 exemples par mise à jour** ;
- séquences de 1 024 jetons au plus (passage + question + réponse tiennent largement) ;
- taux d'apprentissage 2e-4 (valeur classique en QLoRA), décroissance cosinus, 5 % de préchauffage ;
- `gradient_checkpointing` (moins de mémoire, un peu plus lent) et optimiseur `paged_adamw_8bit` ;
- **3 époques** et évaluation sur le jeu de test à chaque époque (surveiller le surapprentissage).

In [ ]:
import math
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

LORA = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE, GRAD_ACCUM = 4, 4
total_steps = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS

peft_config = LoraConfig(
    **LORA,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, round(0.05 * total_steps)),  # 5 % de préchauffage
    max_length=1024,
    fp16=True,               # T4 : float16
    bf16=False,              # le T4 ne gère pas bfloat16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",     # seul l'adaptateur final est sauvegardé (étape 6)
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
# Poids LoRA en float32 : la précision mixte (fp16) exige des paramètres entraînables en float32
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
print("Types des poids entraînables :", {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})
trainer.model.print_trainable_parameters()

## 5. Entraînement

Environ **10 à 20 minutes** pour 300 exemples et 3 époques sur un T4 (la perte s'affiche toutes les 5 étapes).

In [ ]:
import time

t0 = time.time()
train_result = trainer.train()
print(f"Durée : {(time.time() - t0) / 60:.1f} min · perte finale : {train_result.training_loss:.3f}")
print(f"Pic de mémoire GPU : {torch.cuda.max_memory_allocated() / 1e9:.1f} Go")

### Courbe de perte

La perte d'entraînement doit baisser régulièrement. La perte de test (une valeur par époque) doit baisser
aussi : si elle remonte alors que la perte d'entraînement continue de baisser, le modèle apprend par cœur
(surapprentissage) → garder moins d'époques.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 4))
plt.plot(*zip(*train_points), label="entraînement")
if eval_points:
    plt.plot(*zip(*eval_points), "o-", label="test (fin d'époque)")
plt.xlabel("étape"); plt.ylabel("perte"); plt.title("Fine-tuning LoRA — courbe de perte")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

## 6. Sauvegarde de l'adaptateur

Seuls les poids LoRA sont sauvegardés (quelques dizaines de Mo), pas le modèle de base.

In [ ]:
ADAPTER_DIR = "study-tutor-qwen2.5-1.5b-lora"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!du -sh {ADAPTER_DIR}

## 7. Génération : modèle de base contre modèle affiné

Pour une comparaison équitable :
- **mêmes prompts** (system + user du jeu de test, sans la réponse attendue) ;
- **même température** (0.3, celle du tuteur de l'application) et même graine ;
- le modèle de base est **le même modèle 4 bits avec l'adaptateur désactivé** (`disable_adapter()`).

Environ 5 à 10 minutes (génération par lots de 8).

In [ ]:
import json

GEN = {"max_new_tokens": 512, "temperature": 0.3, "top_p": 0.9, "do_sample": True, "repetition_penalty": 1.05}
BATCH = 8

model = trainer.model
model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"  # génération par lots : les prompts sont alignés à droite

test_rows = [json.loads(line) for line in open("data/test.jsonl", encoding="utf-8")]

def generate_all(rows):
    replies = []
    for start in range(0, len(rows), BATCH):
        batch = rows[start:start + BATCH]
        prompts = [tokenizer.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True)
                   for r in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        torch.manual_seed(42)
        with torch.inference_mode():
            out = model.generate(**inputs, **GEN, pad_token_id=tokenizer.pad_token_id)
        replies += tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{min(start + BATCH, len(rows))}/{len(rows)}", end=" · ")
    print()
    return [r.strip() for r in replies]

with model.disable_adapter():
    base_replies = generate_all(test_rows)
tuned_replies = generate_all(test_rows)

In [ ]:
# Dépôt du Hub lu dans les secrets Colab (icône 🔑) : HF_REPO_ID (ex. Alinajjaa/study-tutor-qwen2.5-1.5b-lora)
# ou, à défaut, HF_USERNAME (le nom du dépôt est alors ajouté automatiquement). Rien à modifier ici.
from google.colab import userdata


def colab_secret(name):
    try:
        return (userdata.get(name) or "").strip() or None
    except Exception:  # secret absent ou accès non autorisé au notebook
        return None


REPO_ID = colab_secret("HF_REPO_ID")
if REPO_ID is None and colab_secret("HF_USERNAME"):
    REPO_ID = f"{colab_secret('HF_USERNAME')}/study-tutor-qwen2.5-1.5b-lora"
assert REPO_ID, "Ajoute le secret Colab HF_REPO_ID (ou HF_USERNAME), autorise-le pour ce notebook, puis relance."
print("Dépôt du Hub :", REPO_ID)

RUN_CONFIG = {
    "base_model": BASE_MODEL,
    "adapter": REPO_ID,
    "quantization": "4 bits nf4 (QLoRA)",
    "lora_r": LORA["r"], "lora_alpha": LORA["lora_alpha"], "lora_dropout": LORA["lora_dropout"],
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "train_size": len(train_ds), "temperature": GEN["temperature"],
}

def export(path, replies, variant):
    with open(path, "w", encoding="utf-8") as f:
        for row, reply in zip(test_rows, replies):
            f.write(json.dumps({
                "id": row["id"], "qid": row["qid"], "type": row["type"], "niveau": row["niveau"],
                "page": row["page"], "prompt": row["messages"][:-1],
                "reference": row["messages"][-1]["content"], "reponse": reply,
                "modele": BASE_MODEL if variant == "base" else f"{BASE_MODEL} + LoRA",
                "config": RUN_CONFIG,
            }, ensure_ascii=False) + "\n")

export("results_base.jsonl", base_replies, "base")
export("results_finetuned.jsonl", tuned_replies, "affine")

# Aperçu : un exemple par niveau
for level in ("debutant", "intermediaire", "avance"):
    i = next(k for k, r in enumerate(test_rows) if r["niveau"] == level)
    print(f"===== {level} — {test_rows[i]['messages'][1]['content'].split('Question de l')[-1][:120]}")
    print("BASE   :", base_replies[i][:500])
    print("AFFINÉ :", tuned_replies[i][:500], "\n")

### Téléchargement des résultats

À copier dans `backend/app/phase4_finetuning/data/`, puis en local :

```
python -m app.phase4_finetuning.evaluate app/phase4_finetuning/data/results_base.jsonl app/phase4_finetuning/data/results_finetuned.jsonl
```

In [ ]:
!zip -qr {ADAPTER_DIR}.zip {ADAPTER_DIR}
for name in ("results_base.jsonl", "results_finetuned.jsonl", "loss_curve.png", f"{ADAPTER_DIR}.zip"):
    files.download(name)

## 8. Publication sur le Hugging Face Hub

Il faut un jeton **en écriture** (https://huggingface.co/settings/tokens). Le plus sûr : l'ajouter dans les
**secrets de Colab** (icône 🔑 à gauche) sous le nom `HF_TOKEN` ; sinon, `login()` le demande. Le dépôt vient des secrets `HF_REPO_ID` ou
`HF_USERNAME` (étape 7).
Le jeton n'est jamais écrit dans le notebook.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

In [ ]:
final_eval = [h["eval_loss"] for h in history if "eval_loss" in h]
MODEL_CARD = f"""---
base_model: {BASE_MODEL}
library_name: peft
language: fr
license: apache-2.0
tags: [lora, qlora, peft, education, tutoring, french]
---

# Tuteur de cours adapté au niveau — adaptateur LoRA pour Qwen2.5-1.5B-Instruct

## Objectif
Adaptateur LoRA qui apprend à {BASE_MODEL} le style du tuteur de l'application *AI Study Assistant* :
répondre à une question d'étudiant **uniquement à partir d'un passage de son cours**, en citant la page
(`[p.X]`), et en **adaptant l'explication au niveau** indiqué :
- **débutant** : mots simples, termes techniques définis, analogie de la vie courante et exemple concret ;
- **intermédiaire** : explication claire avec les termes du cours et un exemple ;
- **avancé** : réponse précise et technique, sans définitions de base, notions reliées entre elles.

## Données
{len(train_ds)} exemples d'entraînement et {len(eval_ds)} de test (questions jamais vues à l'entraînement),
en français. Questions (définition, rôle, comparaison, exemple, « pourquoi ») et réponses idéales générées
par un modèle enseignant (qwen2.5:7b) à partir de passages de cours, puis filtrées automatiquement (français,
longueur adaptée au niveau, référence de page valide, pas d'hésitation). Aucune donnée d'étudiant ; emails et
numéros masqués.

Format : conversation `system` (règles du tuteur) / `user` (`Niveau de l'étudiant : …` + extraits du cours +
question) / `assistant` (réponse). La perte n'est calculée que sur la réponse.

## Hyperparamètres
| Paramètre | Valeur |
|---|---|
| Quantification | 4 bits nf4, double quantification, calcul en float16, poids LoRA en float32 (QLoRA) |
| LoRA | r={LORA["r"]}, alpha={LORA["lora_alpha"]}, dropout={LORA["lora_dropout"]} |
| Modules | q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj |
| Époques | {EPOCHS} |
| Taux d'apprentissage | {LEARNING_RATE} (cosinus, 5 % de préchauffage) |
| Lot effectif | 16 (4 × 4 accumulations) |
| Longueur max | 1 024 jetons |
| Matériel | Google Colab, GPU T4 (15 Go) |
| Perte finale | entraînement {train_result.training_loss:.3f}{f", test {final_eval[-1]:.3f}" if final_eval else ""} |

## Utilisation
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("{BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}", device_map="auto")
model = PeftModel.from_pretrained(model, "{REPO_ID}")
```

## Limites
- Petit modèle (1,5 milliard de paramètres) : raisonnement limité, erreurs factuelles possibles hors du passage.
- Données générées par un modèle enseignant : le modèle imite ses qualités **et ses défauts** ; le jeu est petit.
- Évaluation par un juge LLM (qwen2.5:7b) : notes indicatives, biais possibles (longueur, style).
- Entraîné pour le français et pour des cours d'informatique ; ne remplace pas un enseignant.
"""

with open(f"{ADAPTER_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(MODEL_CARD)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=REPO_ID, commit_message="Adaptateur LoRA du tuteur")
api.upload_file(path_or_fileobj="loss_curve.png", path_in_repo="loss_curve.png", repo_id=REPO_ID)
print(f"Publié : https://huggingface.co/{REPO_ID}")

## Et ensuite ?

- Évaluer en local : `python -m app.phase4_finetuning.evaluate …` (juge qwen2.5:7b + MLflow).
- Démonstration dans l'application (optionnel) : `LLM_PROVIDER=hf-local` et
  `HF_ADAPTER=<dépôt du Hub>` dans `backend/.env` (voir le README du dossier).